## **Data Embedding**

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model=SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
print(type('model'))

<class 'str'>


In [ ]:
sample=df1["paper_text"].iloc[0]
sample

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate functions. The ideas are\nillustrated on the example of nonparam

In [ ]:
df1["paper_text"]=df1["paper_text"].str.replace("\n"," ",regex=False)
df1["paper_text"]=df1["paper_text"].str.strip()

/tmp/ipykernel_573/2200328001.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["paper_text"]=df1["paper_text"].str.replace("\n"," ",regex=False)
/tmp/ipykernel_573/2200328001.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["paper_text"]=df1["paper_text"].str.strip()


In [ ]:
embeeding=model.encode(sample)

In [ ]:
print(type(embeeding))

<class 'numpy.ndarray'>


In [ ]:
print(embeeding.shape)

(384,)


In [ ]:
embeeding[:50]

array([-0.1315641 , -0.00678266, -0.00367612,  0.03265158,  0.11219642,
        0.01227267,  0.09816719, -0.0900523 ,  0.04231161, -0.01977348,
       -0.03308417,  0.07452948,  0.10632038, -0.02060429, -0.02052106,
        0.00169493,  0.07081953,  0.05854454, -0.11231912,  0.02082474,
        0.05692544,  0.0201578 ,  0.0258311 ,  0.0321703 ,  0.10513764,
       -0.09676763,  0.02700802, -0.0234509 , -0.04549678, -0.01013699,
       -0.01794855, -0.04814427,  0.01077652, -0.03759069,  0.01943481,
        0.03715189,  0.02967844,  0.04330941,  0.04373213,  0.03704866,
       -0.00182594,  0.00455183, -0.00799067,  0.03037368, -0.014378  ,
        0.03795147,  0.0595916 , -0.02583356, -0.06521576,  0.05900268],
      dtype=float32)

In [ ]:
sample_embed=model.encode(df1["paper_text"].head(5).to_list())

In [ ]:
sample_embed.shape

(5, 384)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity=cosine_similarity(sample_embed[0].reshape(1,-1),sample_embed[1].reshape(1,-1))
print(similarity)

[[0.36625272]]


In [ ]:
for i in range(1,5):
  sim=cosine_similarity(sample_embed[0].reshape(1,-1),sample_embed[i].reshape(1,-1))
  print(sim)

[[0.36625272]]
[[0.33522844]]
[[0.15505108]]
[[0.37421533]]


## **Generate Full embedding**

In [ ]:
if os.path.exists("paper_embeddings.npy"):
  print("Loading existing embedding")
  embeddings=np.load("paper_embeddings.npy")
else:
  print("Generating New")
  embeddings=model.encode(df1["paper_text"].to_list(),batch_size=32,show_progress_bar=True)
  np.save("paper_embeddings.npy", embeddings)
  print("Embeddings saved successfully")

Generating New


Batches:   0%|          | 0/469 [00:00<?, ?it/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [ ]:
print(embedding.shape)
print(type(embedding))
embedding.dtype

(15000, 384)
<class 'numpy.ndarray'>


dtype('float32')

## **Normalizing using Faiss**

In [ ]:
import faiss

In [ ]:
if os.path.exists("paper_faiss.index"):
  print("Loading existing")
  index=faiss.read_index("paper_faiss.index")
else:
  print("Creating new faiss index")
  faiss.normalize_L2(embeddings)
  index=faiss.IndexFlatIP(384)
  index.add(emdeddings)

  faiss.write_index(index,"paper_faiss.index")
  print("Faiss index saved successfully")

In [ ]:
faiss.normalize_L2(embedding)

In [ ]:
index=faiss.IndexFlatIP(384)

In [ ]:
index.add(embedding)

In [ ]:
query="deep learning for medical image analysis"
query_embedding=model.encode([query])
query_embedding.shape

(1, 384)

In [ ]:
faiss.normalize_L2(query_embedding)

In [ ]:
D,I=index.search(query_embedding,5)
print(D)
print(I)

[[0.6807244  0.67092204 0.65219975 0.62811744 0.61311525]]
[[10466 13730 11873 12691 11282]]


In [ ]:
print(df.iloc[10466]["title"])

In [ ]:
def search_paper(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print("Similarity score", score)
        print("Title", df.iloc[idx]["title"])
        print("Title", df.iloc[idx]["abstract"][:500])
        print()

search_paper("deep learning for medical image analysis")


In [ ]:
!pip install transformers==4.46.3

In [ ]:
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

type(summarizer)

summary = summarizer(df.iloc[10466]["abstract"], max_length=120, min_length=40)
print(summary)